# Modèles de Machine Learning classiques

In [17]:
import pandas as pd
import numpy as np

In [3]:
# Chargement du dataset
df = pd.read_csv('../data/df_processed.csv')
df.head()

,title,text,subject,date,label,title_length,text_length,text_processed,title_processed,word_count,flesch_score,flesch_kincaid,caps_ratio,exclamation_count,question_count
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",Fake,79,2893,donald trump wish american happi new year leav...,donald trump send embarrass new year eve messa...,495,62.715000,8.726552,0.047701,6,9
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",Fake,69,1898,hous intellig committe chairman devin nune go ...,drunk brag trump staffer start russian collus ...,305,45.677700,11.097678,0.046365,0,0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",Fake,90,3597,friday reveal former milwauke sheriff david cl...,sheriff david clark becom internet joke threat...,580,61.637878,8.657630,0.085627,2,4
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",Fake,78,2774,christma day donald trump announc would back w...,trump obsess even obama name code websit imag,444,54.836522,9.993001,0.044340,0,1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",Fake,70,2346,pope franci use annual christma day messag reb...,pope franci call donald trump christma speech,420,62.447143,9.766190,0.026854,0,0


#### Extraction de caractéristiques

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorisation TF-IDF
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.8,
    ngram_range=(1, 2)  # unigrams et bigrams
)

# Combiner titre et texte, en gérant les valeurs manquantes
df['combined_text'] = (
    df['title_processed'].fillna('') + ' ' + df['text_processed'].fillna('')
)

# Remplacer les valeurs manquantes éventuelles dans la colonne combinée
combined_text = df['combined_text'].fillna('')

X_tfidf = tfidf.fit_transform(combined_text)

print(f"Shape des features TF-IDF: {X_tfidf.shape}")

Shape des features TF-IDF: (44898, 5000)


#### 1- Modèle avec features engineered

In [9]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

In [10]:
# Features numériques
feature_cols = ['text_length', 'word_count', 'flesch_score', 
                'flesch_kincaid', 'caps_ratio', 'exclamation_count', 'question_count']

X_features = df[feature_cols].fillna(0)
y = df['label']

# Division train/test
X_train_feat, X_test_feat, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42, stratify=y
)

# Entraînement Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_feat, y_train)

# Prédictions
y_pred = rf_model.predict(X_test_feat)
y_prob = rf_model.predict_proba(X_test_feat)[:, 1]  # Probabilité classe positive

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.877

Classification Report:
              precision    recall  f1-score   support

        Fake       0.88      0.88      0.88      4696
        Real       0.87      0.87      0.87      4284

    accuracy                           0.88      8980
   macro avg       0.88      0.88      0.88      8980
weighted avg       0.88      0.88      0.88      8980



#### 2- Modèle TF-IDF + Logistic Regression

In [11]:
# Division des données TF-IDF
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

# Prédictions
y_pred_lr = lr_model.predict(X_test_tfidf)
y_prob_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]

print(f"TF-IDF + LR Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")

TF-IDF + LR Accuracy: 0.989


#### 3-Modèle Transformer (BERT)

##### 3.1 - Préparation pour BERT

In [12]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm

In [13]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [14]:
# Initialisation du modèle BERT
model_name = 'bert-base-uncased'  # ou 'camembert-base' pour le français
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
from sklearn.preprocessing import LabelEncoder
#Conversion des labels en entiers
def prepare_labels(df):
    """Convertit les labels en format numérique"""
    if df['label'].dtype == 'object':  # Si c'est des strings
        le = LabelEncoder()
        df['label_encoded'] = le.fit_transform(df['label'])
        print(f"Mapping des labels: {dict(zip(le.classes_, le.transform(le.classes_)))}")
        return df['label_encoded'].values, le
    else:
        return df['label'].values, None

##### 3.2 Entraînement BERT3

In [ ]:
# Préparation des données - CORRECTION COMPLÈTE
print("Préparation des données...")

# 1. Conversion des labels
y_encoded, label_encoder = prepare_labels(df)

# 2. Préparation des textes
X_text = df['combined_text'].fillna('').values

# 3. Division train/test
X_train_text, X_test_text, y_train_bert, y_test_bert = train_test_split(
    X_text, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Taille train: {len(X_train_text)}, Taille test: {len(X_test_text)}")
print(f"Distribution labels train: {np.bincount(y_train_bert)}")

# 4. Créer les datasets pour BERT
train_dataset = NewsDataset(X_train_text, y_train_bert, tokenizer)
test_dataset = NewsDataset(X_test_text, y_test_bert, tokenizer)

# Arguments d'entraînement (version corrigée)
training_args = TrainingArguments(
    output_dir='../results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='../logs',
    eval_strategy="epoch",  # Changé de evaluation_strategy à eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to=None,  # Désactive les logs WandB/TensorBoard si non configurés
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)

# Entraînement
trainer.train()

Préparation des données...
Mapping des labels: {'Fake': np.int64(0), 'Real': np.int64(1)}
Taille train: 35918, Taille test: 8980
Distribution labels train: [18785 17133]


/var/folders/xr/92zkzk9s5b59pq50n34_ny6c0000gn/T/ipykernel_5067/2013262567.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/davidamouzou/codex/Fake_news/venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Sauvegarde
model.save_pretrained('./fake-news-bert')
tokenizer.save_pretrained('./fake-news-bert')